# v12 Notebook

Notebook version of `v12.py`.

The temporal rapidity uses the fixed constant `L = 1/1024`, while the spatial inputs are used directly.

In [38]:
from __future__ import annotations
import numpy as np

ETA4 = np.diag([-1.0, 1.0, 1.0, 1.0]).astype(np.float64)
L = (2*np.pi) / 1024.0


def minkowski_dot_big_vec(u: np.ndarray, v: np.ndarray) -> float:
    U = u.reshape(-1, 4)
    V = v.reshape(-1, 4)
    return np.sum((U @ ETA4) * V)


class TriadMonSTERFastVec:
    def __init__(
        self,
        dim: int = 768,
        base: float = 10000.0,
    ):
        if dim % 12 != 0:
            raise ValueError(f'dim must be divisible by 12, got {dim}.')
        self.dim = dim
        self.num_freq = dim // 12
        self.base = float(base)
        self.L = L
        j = np.arange(self.num_freq, dtype=np.float64)
        self.inv_freq = self.base ** (-j / self.num_freq)
        self._cache = {}

    def forward(self, s: np.ndarray):
        s = np.asarray(s, dtype=np.float64)
        if s.shape != (4,):
            raise ValueError('position must be a 4D vector (t, x, y, z).')
        key = (s[0], s[1], s[2], s[3])
        if key in self._cache:
            return self._cache[key]

        t, x, y, z = s
        lam = self.inv_freq

        phi = (self.L * t) * lam
        thx = x * lam
        thy = y * lam
        thz = z * lam

        ch = np.cosh(phi)
        sh = np.sinh(phi)
        c_axes = np.stack((np.cos(thx), np.cos(thy), np.cos(thz)), axis=1)
        s_axes = np.stack((np.sin(thx), np.sin(thy), np.sin(thz)), axis=1)

        out = {'ch': ch, 'sh': sh, 'c': c_axes, 's': s_axes}
        self._cache[key] = out
        return out


def apply_monster_triad_fast_vec(emb: np.ndarray, tables: dict, dim: int = 768) -> np.ndarray:
    if emb.shape != (dim,):
        raise ValueError(f'embedding must be shape ({dim},), got {emb.shape}')
    F = dim // 12

    V = emb.reshape(F, 3, 4).astype(np.float64, copy=False)
    out = V.copy()

    ch = tables['ch']
    sh = tables['sh']
    c_axes = tables['c']
    s_axes = tables['s']

    comp_idx = np.array([1, 2, 3], dtype=np.int64)[None, :, None]
    t = out[:, :, 0]
    x_axis = np.take_along_axis(out, comp_idx, axis=2)[..., 0]

    t1 = ch[:, None] * t - sh[:, None] * x_axis
    x1 = -sh[:, None] * t + ch[:, None] * x_axis

    out[:, :, 0] = t1
    np.put_along_axis(out, comp_idx, x1[..., None], axis=2)

    pair_idx = np.array([[2, 3], [1, 3], [1, 2]], dtype=np.int64)[None, :, :]
    pair_vals = np.take_along_axis(out, pair_idx, axis=2)
    u = pair_vals[..., 0]
    v = pair_vals[..., 1]

    u2 = c_axes * u - s_axes * v
    v2 = s_axes * u + c_axes * v

    rotated = np.stack((u2, v2), axis=-1)
    np.put_along_axis(out, pair_idx, rotated, axis=2)

    return out.reshape(dim,)


In [39]:
np.random.seed(0)
DIM = 768
monster = TriadMonSTERFastVec(dim=DIM, base=10000.0)

s_q = np.array([700.0, 500.0, -300.0, 200.0], dtype=np.float64)
s_k = np.array([-40.0, -20.0, 60.0, -10.0], dtype=np.float64)
dskq = s_k - s_q

T_abs_q = monster.forward(s_q)
T_abs_k = monster.forward(s_k)
T_rel = monster.forward(dskq)

q = np.random.uniform(-0.6, 0.6, size=DIM).astype(np.float64)
k = np.random.uniform(-0.6, 0.6, size=DIM).astype(np.float64)

q_abs = apply_monster_triad_fast_vec(q, T_abs_q, dim=DIM)
k_abs = apply_monster_triad_fast_vec(k, T_abs_k, dim=DIM)

lhs = minkowski_dot_big_vec(q_abs, k_abs)
k_rel = apply_monster_triad_fast_vec(k, T_rel, dim=DIM)
rhs = minkowski_dot_big_vec(q, k_rel)

print('RoPE-style identity holds? ', np.allclose(lhs, rhs, rtol=1e-12, atol=1e-12))
print(f'lhs: {lhs:+.12f}  rhs: {rhs:+.12f}')

Q_blocks = q.reshape(-1, 4)
Q_abs_blocks = q_abs.reshape(-1, 4)
norms_before = np.sum((Q_blocks @ ETA4) * Q_blocks, axis=1)
norms_after = np.sum((Q_abs_blocks @ ETA4) * Q_abs_blocks, axis=1)
ok = np.allclose(norms_before, norms_after, rtol=1e-11, atol=1e-12)
max_err = np.max(np.abs(norms_before - norms_after))
print('Per-4D Minkowski norms preserved? ', ok, '| max abs err:', max_err)

print('NUM_FREQ:', DIM // 12, ' | DIM:', DIM, ' | BLOCK dim:', 12)


RoPE-style identity holds?  True
lhs: -15.868385081431  rhs: -15.868385081431
Per-4D Minkowski norms preserved?  True | max abs err: 1.0624834345662748e-13
NUM_FREQ: 64  | DIM: 768  | BLOCK dim: 12


In [40]:
random_vec = np.random.default_rng().uniform(-0.6, 0.6, size=DIM).astype(np.float64)
random_vec_self_dot = minkowski_dot_big_vec(random_vec, random_vec)

print(f'Random 768D vector Minkowski self-dot: {random_vec_self_dot:+.12f}')


Random 768D vector Minkowski self-dot: +46.179778654774


In [ ]:
q_t, q_x, q_y, q_z = 1000, 64, 64, 64
k_t, k_x, k_y, k_z = 0, 0, 0, 0

query = random_vec.copy()
key = random_vec.copy()
dim = random_vec.size
monster_abs = TriadMonSTERFastVec(dim=dim, base=10000.0)

query_pos = np.array([q_t, q_x, q_y, q_z], dtype=np.float64)
key_pos = np.array([k_t, k_x, k_y, k_z], dtype=np.float64)

query_abs = apply_monster_triad_fast_vec(query, monster_abs.forward(query_pos), dim=dim)
key_abs = apply_monster_triad_fast_vec(key, monster_abs.forward(key_pos), dim=dim)

original_query_key_dot = minkowski_dot_big_vec(query, key)
encoded_query_key_dot = minkowski_dot_big_vec(query_abs, key_abs)

print(f'Original query/key Minkowski dot: {original_query_key_dot:+.12f}')
print(f'Encoded query/key Minkowski dot:  {encoded_query_key_dot:+.12f}')


Original query/key Minkowski dot: +46.179778654774
Encoded query/key Minkowski dot:  +95.968225761543
